# Player Retention and Churn Modeling
*This notebook analyzes player behavior from a simulated multiplayer game.*

Notebook Author: Ilan Harari <br>
Data Source: https://pypi.org/project/players-behaviors-dataset-generator/#description 

## Project Planning

**Project Title:**  
Early Life Cycle Player Engagement & Retention (Simulated Launch Analysis)

**Context:**
This is a simulated analysis focused on early engagement, retention curve shaping and segmentation for re-engagement. 

**Problem Statement:**  
In the early stages of a feature launch, certain players will churn, this is known as player funnel abandonment. This is a key engagement phenomenon to understand and incredibly powerful. When players disengage early, this is a key opportunity to understand glaring issues in the player experience, as well as address what can ultimately become major loss in revenue potential. This project models early session behavior to identify churn risk patterns and missed monetization windows. With understanding of where and why players disengage, product teams are empowered to deploy targeted re-engagement strategies that extend player longevity and increase revenue coverage per acquisition. 

**Primary Objective:**  
Build an interpretable & actionable churn model and player segmentation framework to support retention strategy.

**Key Business Questions:**
- When and how are players most active?
- What types of players are more likely to churn?
- How can we segment users meaningfully for targeted interventions?
- What feature usage or gameplay metrics correlate with longer retention?

**Success Criteria:**
- Identify churn-prone player segments using behavior-driven clustering
- Achieve >75% recall in churn classification (initial benchmark)
- Propose testable re-engagement strategy for early life cycle players likely to churn

**Metrics:**
- North Star: D1/D7/D30 retention
- Supporting: session count, avg session length, active days

**Constraints:**
- No real user sign-up timestamp (must infer activity from event timing)
- Synthetic dataset with potential assumptions baked into distributions
- Simplified modeling and experimentation scope due to time constraints
- Synthetic data may not reflect real funnel dynamics (no drop-off curves)
- No actual monetization behavior; can optionally simulate with rules tied to progression
- No multi-device or account linking — player_id assumed 1:1 with user

**Planned Features to Engineer:**
- Session count per player
- Event type distribution
- Time between events (lag features)
- Active days / time since first event
- Time of day and day-of-week behavior
- Average stage score and stage engagement

#### Imports and Data Loading

In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import os
print("All libraries are working!")

In [ ]:
#read in data
df = pd.read_csv('../data/synthetic_player_data.csv')

## Exploratory Data Analysis

### Data Validation & Cleaning



- Data Validation & Cleaning    
    - Assessment of column data types, nulls, duplicates, outliers
    - Column standardization
    - Descriptive Statistics
    - Findings
- Feature Engineering
    -  Extraction
    -  Transformation
    -  Derivation
- Post-Cleaning Validation
    - Validation Report
- Visualizations
    - Time series plots
    - Cohort plots
    - Session plots

#### Assessment of column data types, nulls, duplicates, outliers

In [71]:
df.columns

Index(['id', 'cohort_id', 'player_id', 'player_type', 'session_id',
       'event_type', 'timestamp', 'stage_id', 'stage_score', 'session_length',
       'events_per_session', 'sessions_per_player', 'weekday', 'hour', 'week',
       'day', 'events_per_player'],
      dtype='object')

In [ ]:
df.shape
df.info()
df.head()

In [ ]:
# 1. Null check
print("\nNull raw:\n",df.isnull().sum().sort_values(ascending=False).head()) #raw null count
print("\nNull rate:\n", df.isnull().sum() / len(df) * 100) #null rate

In [ ]:
# Explore null stage score and stage id
print(df[df['stage_score'].isna()]['event_type'].value_counts())
print(df[df['stage_id'].isna()]['event_type'].value_counts())

In [ ]:
#2. Duplicate check
df.duplicated().sum()

In [ ]:
# Outlier check - I'm not going to do this yet because I'm not sure if the outliers are valid

# #impute stage_score with upper limit for outliers
# upper_limit = df['stage_score'].quantile(0.75) + 1.5 * (df['stage_score'].quantile(0.75) - df['stage_score'].quantile(0.25))
# print(upper_limit)
# #impute outliers with upper limit
# df['stage_score'] = np.where(df['stage_score'] > upper_limit, upper_limit, df['stage_score'])
# #check if outliers are removed
# df['stage_score'].describe(percentiles=[.25, .5, .75])

# #proceed with lower limit
# lower_limit = df['stage_score'].quantile(0.25) - 1.5 * (df['stage_score'].quantile(0.75) - df['stage_score'].quantile(0.25))
# print(lower_limit)
# #impute outliers with lower limit
# df['stage_score'] = np.where(df['stage_score'] < lower_limit, lower_limit, df['stage_score'])
# #check if outliers are removed
# df['stage_score'].describe(percentiles=[.25, .5, .75])

#### Column standardization

In [ ]:
#3. Fix timestamp type
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'].head()

In [ ]:
#add weekday and hour columns
df['weekday'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

#add week column
df['week'] = df['timestamp'].dt.isocalendar().week

#add day column 
df['day'] = df['timestamp'].dt.day

#check the new columns
df[['timestamp', 'week', 'weekday', 'day', 'hour']].head()

In [ ]:
#4. Rename columns
df.rename(columns={'id': 'event_id', 'player_type':'player_label', 'cohort_id': 'player_cohort'})

#### Descriptive Statistics

In [ ]:
df.describe(include='object')

In [ ]:
#Explore stage score
df['stage_score'].describe(percentiles=[.25, .5, .75, .9, .95, .99, .991, .995])

*Data has been scrubbed for nulls and duplicates, columns renamed for consistency, and datetime typed appropriately.*



#### Findings



- For a churn prediction model, the target variable/label will be `player_label`, which has 3 unique values: `churned`, `casual`, and `hardcore`.

- There are 2 columns, `stage_id` and `stage_score` which have null rates of 4% and 52%, respectively. Upon aggregation of these columns with `event_type`, it is apparent that `stage_score` is null only when a player is starting & ending a session, or starting a stage. This is valid and does not warrant imputation. `stage_id` is null only for event types of `BEGIN_SESSION` and `END_SESSION`, which also makes logical sense. No imputation is needed here.

- `stage_score` has many outliers. I'm not ready to impute them yet, would like to know more about those outliers.

- There are 16 unique cohorts of players. Their names are formatted as unique identifiers, which does not clue in as to how they are formed or what they represent. My exploratory analysis will address this to uncover any patterns within and across cohorts.

### Feature Engineering

- Feature Selection
- Feature Extraction
- Feature Transformation


#### Feature Selection

#### Feature Extraction

#### Feature Transformation

In [ ]:
#Calculate events per session
df['events_per_session'] = df.groupby('session_id')['event_type'].transform('count')

In [ ]:
#Calculate session length, matching session id
df['session_length'] = df.groupby('session_id')['timestamp'].transform(lambda x: x.max() - x.min())

In [ ]:
df['events_per_session'].describe()


In [ ]:
#calculate sessions per player
df['sessions_per_player'] = df.groupby('player_id')['session_id'].transform('nunique')

In [ ]:
df['sessions_per_player'].describe()

In [ ]:
#calculate events per player
df['events_per_player'] = df.groupby('player_id')['id'].transform('nunique')

In [ ]:
# DateTime columns
df['weekday'] = df['timestamp'].dt.strftime('%A')
df['hour'] = df['timestamp'].dt.strftime('%H')
df['minute'] = df['timestamp'].dt.strftime('%M')


### Visualizations


- *insert here what is contained in this section.*

In [ ]:
# Drop duplicates so each player is counted once
player_sessions = df.drop_duplicates('player_id')['sessions_per_player']

# Create 5 percentile-based bins (quintiles as example)
sessions_bins = pd.qcut(player_sessions, q=5)

# Count players in each bin
sessions_dist = sessions_bins.value_counts().sort_index()

# Plot
sessions_dist.plot(kind='bar')
plt.xlabel('Sessions per Player (Quantile Bins)')
plt.ylabel('Number of Players')
plt.title('Distribution of Sessions per Player (Percentile-based Bins)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Drop duplicates so each player is counted once
player_events = df.drop_duplicates('player_id')['events_per_player']

# Create 5 percentile-based bins (quintiles as example)
events_bins = pd.qcut(player_events, q=5)

# Count players in each bin
event_dist = events_bins.value_counts().sort_index()

# Plot
event_dist.plot(kind='bar')
plt.xlabel('Events per Player (Quantile Bins)')
plt.ylabel('Number of Players')
plt.title('Distribution of Events per Player (Percentile-based Bins)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# target variable
sns.histplot(data=df, x='player_type')
plt.title("Count of events by Player Type")

In [ ]:
#Events by day
df.set_index('timestamp')['event_type'].resample('D').count().plot()
plt.title("Events Over Time (Daily)")
plt.ylabel("Number of Events")

In [ ]:
# Events by Week
df.set_index('timestamp')['event_type'].resample('W').count().plot()
plt.title("Events Over Time (Weekly)")
plt.ylabel("Number of Events")

In [ ]:
order = list(range(0, 24))
df['hour'] = df['hour'].astype(int)
sns.countplot(data=df, x='hour', order=order, )
plt.title("Events by Hour of Day")
plt.xlabel("Hour (0 = Midnight)")

In [ ]:
# Events by day of week
order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
sns.countplot(data=df, x='weekday', palette='muted', order=order)
plt.title("Events by Day of Week")
plt.xlabel("Day (0 = Sunday)")
plt.xticks(rotation=45)

In [ ]:
df.columns

In [ ]:
# cohort representation over time
df['week'] = df['timestamp'].dt.strftime('%U')
#make a catplot showing top 3 cohorts for each week of the month

weekly_counts = df.groupby(['week', 'cohort_id'])['player_id'].count().reset_index()

weekly_counts.columns = ['week', 'cohort_id', 'event_count']

top3 = weekly_counts.sort_values(['week', 'event_count'], ascending=[True, False])
top3 = top3.groupby('week').head(3)


In [ ]:
top3 = top3.groupby(['week', 'cohort_id']).sum()

In [ ]:
top3

In [ ]:

sns.catplot(data=top3, kind='bar', x='week', y='event_count', hue='cohort_id')

#make a catplot showing top 3 cohorts for each day of the week 

#make a catplot showing top 3 cohorts for each hour of the day 


In [ ]:
unique_players_by_type = df.groupby('player_type')['player_id'].nunique()

In [ ]:
print(unique_players_by_type)

In [ ]:
unique_players_by_cohort = df.groupby('cohort_id')['player_id'].nunique()


In [ ]:
#target label
print("Target label raw counts:\n", df['player_type'].value_counts())
print("\nTarget label percentages:\n", df['player_type'].value_counts(normalize=True))

In [ ]:
df.columns

In [ ]:
# view player type by cohort
type_by_cohort = df.groupby(['cohort_id', 'player_type'])['session_id'].count()


In [ ]:
type_by_cohort

In [ ]:
pivot = df.pivot_table(
    index='cohort_id',
    columns='player_type',
    values='player_id',
    aggfunc='nunique',   # Or 'count' if each row = 1 player
    fill_value=0         # Optional: replace NaNs with 0s
)

In [ ]:
pivot

## Statistical Testing

*PLACEHOLDER:*
- Hypothesis Test: Player Churn by Last Event Type
- Hypothesis Test: Player Churn by Stage ID

## Predictive Modeling

### Model Construction

- Model Selection
- Training & Validation Dataset formation
- Hyperparameter Selection & Tuning
- Model Estimation
- Model Fitting

#### Model Selection

#### Training & Validation Dataset Formation

#### Hyperparameter Selection & Tuning

#### Model Estimation

#### Model Fitting

### Model Evaluation

- Model Scoring

#### Model Scoring